In [4]:
## important: on simule un utilsateur qui navigue sur agatha
## 
## le fonctionnement est le suivant: on doit faire une requete http pour charger
## la page et (surtout) pour récuperer les cookies de session qui vont bien et 
## qui stockent notre requete. Ensuite on peut faire appel au webservice "ajax.php"
## qui permet d'effectivement récupérer les informations qu'on cherche dans les
## databases des archives.
##
## Pour bien piger comment ca marche, il est utile d'aller jeter un oeil au notebook
## qui s'appelle "en_mode_gros_bourrin" car c'est ici que j'ai enregistré les infos
## utiles (tq un prototype de schéma json pour les requetes et reponses).
import bs4
import requests as rq
import pandas as pd
import pydantic

from typing import Annotated, Literal

In [23]:
AJAX_URL  = "https://agatha.arch.be/search/genealogie/ajax.php"
GENEA_URL = "https://agatha.arch.be/search/genealogie/results/"

GenealogySource = Literal[
    Annotated["PR", "Parish Register before 1796"],
    Annotated["CS", "Civil State after 1796"]
]

class Municipality(pydantic.BaseModel):
    """All the technical data used to retrieve a given municipality"""

    label:         Annotated[str, "How this municipality should be displayed"]
    value:         Annotated[str, "Name that should be used when referencing this municipality"]
    CURRENT_NAME:  Annotated[str, "The current denomincation of this municipality"]
    PLACEPOINT_ID: Annotated[int, "The effective identifier of this municipality"]
    PARISHES_ID:   Annotated[int | None, "If present, the id of the parish in the municipality"]


def municipality_by_name(
    name:   str, 
    source: GenealogySource = "CS"
) -> list[Municipality]:
    """Find municipality data by name."""
    r = rq.post(AJAX_URL, data = {
        "method": "getMunicipalities",
        "type"  : source, # "CS" si on veut l'état civil apres 1796
        "term"  : name,   # location qu'on veut trouver
    })
    return [ Municipality(**x) for x in r.json() ]

In [24]:
x = municipality_by_name("sterrebeek", "CS")

In [25]:
x[0].value

'Sterrebeek (Zaventem)'